## Example for extracting data for GPT prompting

### requires python >= 3.10

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time

In [2]:
pd.set_option('display.max_colwidth', None)

## Vastustega df

In [3]:
df1 = pd.read_csv("../gpt_output/n80_examples_large_v1_gpt_v1_10K_b12_v1.csv", encoding="utf-8", sep="|")

In [4]:
df1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
1,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,NaN
2,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN,yes,NaN
3,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
4,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN,yes,NaN
9996,21044230,Texases,Texas,tulistama,NaN,in,13152394,"Cheney jahilembus pääses tänavu ka meediasse , kui ta Texases kogemata oma linnujahikaaslast tulistas .",NaN,location,LOC,yes,NaN
9997,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
9998,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."


### statistikat

In [74]:
counts2 = df1.groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,count
162,pidama,NaN,149
185,ronima,NaN,117
143,ootama,NaN,110
90,kõndima,NaN,107
57,kaduma,NaN,102
...,...,...,...
106,lendama,tagasi,6
250,turustama,NaN,5
268,viima,edasi,5
228,tarnima,NaN,5


In [6]:
aggreg = df1.groupby(["verb", "verb_compound"], dropna=False).agg(
    yes_count=("classification", lambda x: np.sum(x == "yes")/len(x)*100),
        size = ("classification", lambda x :len(x))
).sort_values('yes_count', ascending=False)
aggreg

,,yes_count,size
verb,verb_compound,,
kärgatama,NaN,100.000000,14
surema,NaN,100.000000,19
eksportima,NaN,100.000000,9
tarnima,NaN,100.000000,5
pussitama,NaN,100.000000,12
...,...,...,...
võtma,välja,29.545455,44
olema,ära,27.500000,40
paistma,NaN,26.829268,41


In [7]:
aggreg[aggreg["yes_count"]>= 80]

,,yes_count,size
verb,verb_compound,,
kärgatama,NaN,100.0,14
surema,NaN,100.0,19
eksportima,NaN,100.0,9
tarnima,NaN,100.0,5
pussitama,NaN,100.0,12
...,...,...,...
pidutsema,NaN,80.0,20
konutama,NaN,80.0,20
külmuma,NaN,80.0,25


In [8]:
aggreg[aggreg["yes_count"]<= 30]

,,yes_count,size
verb,verb_compound,,
tekkima,NaN,29.787234,47
võtma,välja,29.545455,44
olema,ära,27.500000,40
paistma,NaN,26.829268,41
kirjutama,NaN,23.404255,47
voolama,välja,14.285714,21


In [45]:
ab = df1.loc[df1["verb"]=="voolama", ["form", "sentence", "classification", "explanation"]]
ab

,form,sentence,classification,explanation
74,pirukast,Paljudel moosipirukasõpradel on ühine mure : täidis voolab küpsemise käigus pirukast välja .,no,"The phrase 'pirukast' does not refer to a geographic or physical location but rather to the filling flowing out of a pie, so it is classified as 'no'."
716,hetkest,"3 ) vedeliku hulga , mis laadimiskail voolab välja suurimal laadimiskiirusel laadimisvooliku , laadimisvarre või mõne muu vedeliku pidevaks laadimiseks kasutatava seadme ( laadimisseade ) või torujuhtme täieliku purunemise hetkest kuni juurdevoolu täieliku peatamiseni .",no,"The phrase 'hetkest' refers to a point in time, not a physical location, so it was classified as 'no'."
808,ahjusuust,Põlemis- ja lõhustumisaadused on õhust raskemad ja “ voolavad ” ahjusuust kergesti välja ning põhjustavad ruumis viibijaile raske mürgistuse .,yes,NaN
1245,meditsiinist,Nii voolab riiklikust meditsiinist elujõud muudkui välja .,no,"The word 'meditsiinist' refers to the medical field, which is an abstract concept and not a location."
1982,mänguasjast,"2 ) mänguasjas sisalduvad vedelikud ja gaasid ei saavutaks sellist temperatuuri ega rõhku , et nad voolaksid mänguasjast välja muul juhul , kui on vajalik mänguasja funktsioneerimiseks , ning võiksid tekitada põletuse või muu kehalise vigastuse ohtu .",no,"The phrase 'mänguasjast' refers to a toy and not a geographical location or point, so it is classified as 'no'."
2497,teokarbist,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'."
2769,Muusikamasinast,"Muusikamasinast voolasid välja sajandialguse saksa ja eesti lööklaulud , süüa ei pidanud ka liiga kaua ootama .",no,The phrase 'Muusikamasinast' refers to a source of music and not a location.
3940,kestast,"Paar tundi tules soojenenud ja siis lõhkenud mürsk oleks eridemineerijate hinnangul võinud veelgi traagilisemaid tagajärgi põhjustada , kuid õnneks oli osa lõhkeainet juba enne plahvatust kestast välja voolanud .",no,"The phrase 'kestast' refers to a shell or casing and is not a location, so it was classified as 'no'."
4039,Aukudest,Aukudest voolab maitsev lihamahl välja ja liha jääb kuivem .,no,"The phrase 'Aukudest' refers to holes, which are not locations in a geographic sense."
4899,silmadest,lili: ja voolab välja silmadest.,no,"The phrase 'silmadest' pertains to body parts (eyes) rather than a geographic or specific place, so it is not considered a location."


In [50]:
ab[~ab["explanation"].fillna("").str.contains("physical")]

,form,sentence,classification,explanation
808,ahjusuust,Põlemis- ja lõhustumisaadused on õhust raskemad ja “ voolavad ” ahjusuust kergesti välja ning põhjustavad ruumis viibijaile raske mürgistuse .,yes,NaN
1245,meditsiinist,Nii voolab riiklikust meditsiinist elujõud muudkui välja .,no,"The word 'meditsiinist' refers to the medical field, which is an abstract concept and not a location."
1982,mänguasjast,"2 ) mänguasjas sisalduvad vedelikud ja gaasid ei saavutaks sellist temperatuuri ega rõhku , et nad voolaksid mänguasjast välja muul juhul , kui on vajalik mänguasja funktsioneerimiseks , ning võiksid tekitada põletuse või muu kehalise vigastuse ohtu .",no,"The phrase 'mänguasjast' refers to a toy and not a geographical location or point, so it is classified as 'no'."
2497,teokarbist,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'."
2769,Muusikamasinast,"Muusikamasinast voolasid välja sajandialguse saksa ja eesti lööklaulud , süüa ei pidanud ka liiga kaua ootama .",no,The phrase 'Muusikamasinast' refers to a source of music and not a location.
3940,kestast,"Paar tundi tules soojenenud ja siis lõhkenud mürsk oleks eridemineerijate hinnangul võinud veelgi traagilisemaid tagajärgi põhjustada , kuid õnneks oli osa lõhkeainet juba enne plahvatust kestast välja voolanud .",no,"The phrase 'kestast' refers to a shell or casing and is not a location, so it was classified as 'no'."
4039,Aukudest,Aukudest voolab maitsev lihamahl välja ja liha jääb kuivem .,no,"The phrase 'Aukudest' refers to holes, which are not locations in a geographic sense."
4899,silmadest,lili: ja voolab välja silmadest.,no,"The phrase 'silmadest' pertains to body parts (eyes) rather than a geographic or specific place, so it is not considered a location."
5637,rehvist,"Autojuht tunnistas , et oli oma BMW igasse rehvi mahutanud 30 liitrit piiritust täpsemal uurimisel voolas rehvist välja koguni 33 liitrit alkoholi .",no,"The word 'rehvist' refers to a car tire, which is an object rather than a location."
6475,piludest,Munakollane jääb nõkku pidama ja -valge voolab piludest välja .,no,"The term 'piludest' refers to openings or slots, which are not a defined location."


### ekilex != loc gpt = loc

In [55]:
df1.loc[(df1["ekilex_tag"] != "location") & (df1["classification"]=="yes"), ["form", "sentence", "classification", "ekilex_tag", "explanation"]]

,form,sentence,classification,ekilex_tag,explanation
0,autodesse,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,yes,NaN,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
1,põõsas,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",yes,NaN,NaN
2,vitriinides,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",yes,NaN,NaN
4,töökohta,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",yes,NaN,NaN
5,peatusesse,"Damo ise võrdleb seda rongisõiduga : "" ma pole huvitatud juba möödunud maastike taasnägemisest / : / ootan järgmisesse peatusesse jõudmist , kusjuures eriti lõbus oleks veel jõuda peatusesse , mida pole kaardile märgitud "" .",yes,NaN,NaN
...,...,...,...,...,...
9986,lilleputkast,Sõitsin läbi lilleputkast .,yes,NaN,NaN
9988,hoovi,"Väiksemate korrusmajade elanikud peidavad võimaluse korral oma konteineri hoovi , lukustavad selle ja toovad uuesti välja tühjendamise päeval .",yes,NaN,NaN
9989,Põtalovos,"Kuna Venemaa Pihkva raudtee kasutamiseks ametlikku luba ei andnud , siis kuus tundi Põtalovos seisnud rong haagiti taas veduri külge ning sõit läks tagasi Läti suunas .",yes,NaN,NaN
9990,kärule,Ta ronib oma kärule ja asub teele muinasjutuliselt kaugesse New Yorki .,yes,NaN,NaN


### ekilex=loc gpt != loc

In [56]:
df1.loc[(df1["ekilex_tag"] == "location") & (df1["classification"]=="no"), ["form", "sentence", "classification", "ekilex_tag", "explanation"]]

,form,sentence,classification,ekilex_tag,explanation
101,CNNi,"Saddam Hussein sai CNNi abil igal juhul sõnumi , et USA presidendil ei ole sugugi lihtne teda rünnata .",no,location,"The phrase 'CNNi' is not classified as a location because it refers to a news organization, not a geographic location."
111,kaitsepolitseisse,Püss teatas juhtumist kaitsepolitseisse .,no,location,The phrase 'kaitsepolitseisse' refers to an institution or organization (Security Police) and not a geographical location.
428,ETV-s,"“ See , et Toomas Lepp üldse nii kaua ETV-s peadirektorina askeldas , on väljakutse tervele mõistusele , ” kommenteeris ringhäälingunõukogu liige Andrus Herkel .",no,location,"The term 'ETV-s' refers to an organization (Estonian Television) rather than a geographical location, so it was classified as 'no'."
443,kohvikutest,"Inimõiguste teabekeskuse direktor Aleksei Semjonov väitis järjekordselt , et rahutuste ööl arreteeriti ilmsüüta jalakäijaid , kes kohvikutest ja restoranidest koju suundusid .",no,location,"The word 'kohvikutest' refers to cafes, which could refer to places but in this context, it represents establishments rather than physical locations, so it is classified as 'no'."
478,katlasse,"Küsime : kes andis õiguse teha kerjusteks neid , kes 40-50 aasta vältel tootsid ühisesse katlasse , ehitasid üles ettevõtted , jõujaamad , maanteed .",no,location,"The phrase 'katlasse' refers to a metaphorical concept ('common pot') and not a physical place or location, hence it is classified as 'no'."
...,...,...,...,...,...
9566,haridusametis,"Kaheksa linnaosa peale kokku töötab haridusosakondades kirjade järgi 234 inimest , sellele lisaks üle paarikümne inimese linna haridusametis .",no,location,"The phrase 'haridusametis' refers to an office or administrative body rather than a geographical place, so it was classified as 'no'."
9757,esisesse,"14. minutil ebaõnnestus Ruslan Mussajevi pealelöök , ent pall maandus Transi värava esisesse lompi ning Dmitri Ustritski ennetas üllatunud puurivahti Aleksandr Rjabtshuni .",no,location,"The word 'esisesse' does not indicate a specific geographic location but rather describes a general area or position, so it is classified as 'no'."
9760,eesotsas,"Labour Tony Blairiga eesotsas ajab väga konservatiivset ja ettevõtja-sõbralikku majanduspoliitikat , briti liberaalid on neis asjades märksa vasakpoolsemad .",no,location,"The phrase 'eesotsas' indicates a leading position, not a geographic location, so it is classified as 'no'."
9933,Aita,"Kohe üldse ei tahtnud , mina sain aru küll "" "" Külma pärast , "" ütles Leena ja pani käe ümber Aita kleenukeste õlgade .",no,location,"The word 'Aita' appears to refer to a person's name rather than a location, so it is classified as 'no'."


### gpt yes/no vastuste arv

In [6]:
df1[df1["classification"]=="yes"] # 6474

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
1,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,NaN
2,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN,yes,NaN
4,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN,yes,NaN
5,3305932,peatusesse,peatus,ootama,NaN,ill,2072089,"Damo ise võrdleb seda rongisõiduga : "" ma pole huvitatud juba möödunud maastike taasnägemisest / : / ootan järgmisesse peatusesse jõudmist , kusjuures eriti lõbus oleks veel jõuda peatusesse , mida pole kaardile märgitud "" .",NaN,NaN,NaN,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9992,3866507,prügilasse,prügila,leidma,NaN,ill,2411565,"Ilma teejuhita prügilasse rada ei leia , sest teel puuduvad igasugused viidad .",NaN,location,NaN,yes,NaN
9993,13953644,kruusaaugust,kruusaauk,saama,välja,el,8698768,Bronka saab kruusaaugust välja ja liipab edasi .,NaN,location,NaN,yes,NaN
9994,16644781,rajakattel,rajakate,lamama,NaN,ad,10374293,Eesti parim sportlane Erki Nool lamab Kadrioru staadioni väsinud rajakattel .,NaN,location,NaN,yes,NaN
9995,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN,yes,NaN


In [8]:
df1[(df1["classification"]=="yes") & (~df["explanation"].isna())] # 921

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
12,56138,haiglasse,haigla,jooksma,NaN,ill,32675,"Kui ühel vendadest oli ninaoperatsioon , siis jooksis kogu suguvõsa haiglasse voodiservale tema kätt hoidma .",NaN,location,NaN,yes,"The phrase 'haiglasse' is classified as a location because it refers to a physical place, namely, a hospital."
24,3855195,Kenemast,Kenema,rändama,NaN,el,2404912,"Kenemast rändavad kivid pealinna Freetowni Liibanonist pärit äripartnerile , kes viib teemandid riigist välja .",NaN,NaN,NaN,yes,The phrase 'Kenemast' refers to a specific location (the town of Kenema in Sierra Leone) and is therefore classified as a location ('yes').
38,4643907,Kuubasse,Kuuba,maksma,NaN,ill,2896397,"Tõsi , "" Päikesepüüdja "" on samm edasi eelmisel telehooajal hommikuti eetris olnud Anneli Järveti ja Domina koostöös sündinud reisiülevaatest saates "" Kaunimaks kõikjal "" , kus oli selgelt näha , kes maksis võttegrupi kohalesõidukulud Kuubasse või kuhu iganes .",NaN,location,LOC,yes,"The phrase 'Kuubasse' refers to a specific geographical location (Cuba), so it was classified as 'yes'."
48,21260277,koolis,kool,juhatama,NaN,in,13288680,"Vahel tahavad tööandjad teada nende õppejõudude nimesid , kes kõnealuses koolis teaduskondi ja õppetoole juhatavad .",NaN,NaN,NaN,yes,"The phrase 'koolis' was classified as 'yes' because it specifies a physical place of learning, which qualifies as a location."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9960,12639641,nõmmemetsades,nõmmemets,hulkuma,NaN,in,7895084,Siis hulkusid nad nõmmemetsades .,NaN,location,NaN,yes,"The phrase 'nõmmemetsades' refers to 'moor forests,' which is a type of location or landscape."
9961,7699947,tallu,talu,lubama,NaN,adit,4794600,"Eesti Energia autod keeravad metsateele , aga Nõmme tallu energia-mehed kiiresti elektrit ei luba , see tuleb teist kaudu vedada kui seni .",NaN,NaN,NaN,yes,"The phrase 'tallu' refers to a 'farmstead,' which is a specific type of location or place."
9972,4828792,Pärnus,Pärnu,nappima,NaN,in,3010384,""" Need numbrid näitavad ilmekalt , et Pärnus napib elamukrunte , "" selgitas kinnisvarabüroo LVM juhatuse liige Ingmar Saksing .",NaN,location,LOC,yes,"The term 'Pärnus' refers to a specific geographical place, Pärnu, so it is classified as 'yes'."
9984,1655206,Portugalist,Portugal,lendama,NaN,el,1040882,"Nädalasel Euroopa-ringreisil viibiv USA president Bill Clinton lendas eile Portugalist Saksamaale , kust edasi viib reis ta homseks Moskvasse kohtumisele Vene presidendi Vladimir Putiniga .",NaN,location,LOC,yes,"The phrase 'Portugalist' refers to a specific geographic location, Portugal, and was classified as 'yes'."


In [9]:
df1[df1["classification"]=="no"] # 3334

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
3,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
10,8111391,That's,That,lindistama,NaN,in,5054332,"Elvis , Scotty Moore ja Bill Black lindistasid "" That's all right "" -nimelise laulu Sun Recordsile 5. juulil 1954. aastal .",NaN,NaN,LOC,no,"The term 'That's' refers to part of a song title and not a location, so it is classified as no."
11,3862261,ajakirjandusest,ajakirjandus,kostma,NaN,el,2408977,"Nüüd , kui esimesed kired Eurovisiooni lauluvõistluse järel on vaibumas , kostab mitmete riikide ajakirjandusest hääli , et Melodi Grand Prix vöistlusreegleid tuleks muutma hakata .",NaN,NaN,NaN,no,"The term 'ajakirjandusest' refers to journalism or media and not a physical geographic location, hence it is classified as no."
18,12418343,Liidus,liit,jooma,NaN,in,7746276,"Nõukogude Eestis elas üks nn raamaturahvas , kes seda maad külastanud türgi luuletaja Nazõm Hikmeti sõnul “ luges kõige rohkem luulet ja jõi kõige rohkem viina ” terves Nõukogude Liidus .",NaN,NaN,LOC,no,"The phrase 'Liidus' is not classified as a location because it is part of the name 'Nõukogude Liidus', which refers to an organization (Soviet Union) rather than a specific geographical location."
21,27385004,kaugõppesse,kaugõpe,saama,sisse,ill,17996437,"Jõgeva elanik , kahe lapse ema Katri ( 32 ) sai sisse Tallinna Pedagoogikaülikooli alghariduspedagoogika kaugõppesse .",NaN,NaN,NaN,no,The phrase 'kaugõppesse' is not classified as a location because it refers to a mode of education (distance learning) rather than a physical place.
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9987,11945869,Valgamaalases,Valgamaalane,tutvustama,NaN,in,7440152,"Teost tutvustas 12. augusti Valgamaalases trükise toimetaja Hans Salm , valmimislugu on välja pandud keskraamatukogu teenindussaalis .",NaN,NaN,NaN,no,"The phrase 'Valgamaalases' refers to a publication and not a physical location, so it was classified as 'no'."
9991,12421351,kultuurist,kultuur,pääsema,välja,el,7748164,"Mida teha , et pääseksime Eestis välja sellisestpoliitilisest kultuurist , kus leiavad aset sellised juhtumid nagu viimane näide peaministriga ?",NaN,NaN,NaN,no,"The phrase 'kultuurist' pertains to cultural elements and not a specific location, so it was classified as 'no'."
9997,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
9998,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."


## Näiteid vastustest

In [6]:
df = df1[["form", "lemma", "verb", "verb_compound", "morph_case", "sentence", "classification", "explanation"]]
idx1 = (df["classification"]=="no")

df[idx1 & (df["form"]=="näkku")]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
1237,näkku,nägu,vedama,NaN,adit,"Aastad on Henno pruuniks parkunud näkku vagusid vedanud , metsa all toimetades meenutab ta Eno Raua lasteraamatu kangelast Sammalhabet - vaid linnupesa on veel pikast hallisegusest habemest puudu .",no,"The word 'näkku' refers to a face, which is a part of a body, not a geographical or physical location."
5890,näkku,nägu,mahtuma,ära,adit,"rebis: rõõm on nii suur kohe , ett ei mahu näkku ära",no,"The phrase 'näkku' refers to a face or expression, which is not a geographic location, so it was classified as 'no'."


In [15]:
df[idx1 & (df["form"].str.contains("hinges"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
2052,hinges,hing,helisema,NaN,in,"Minu hinges heliseb lapsepõlves Leida Lepajõelt kuuldud jutustus sellest , kuidas nad , mõned hulljulged Pärnu gümnaasiumi tüdrukud , aiakäru rattad koidikueelsel Pärnu munakivisillutisel tärisemas , käisid päästmas-peitmas äsja õhitud Amandus Adamsoni vabadussamba säilinud detaili , Poiss lilledega ...",no,The word 'hinges' is not a location but rather an abstract concept related to feelings.
3546,hinges,hing,pesitsema,NaN,in,"Coulthardi hinges pesitseb okas mulluse kaotuse pärast tiimikaaslasele , ta januneb revanshi järele .",no,"The word 'hinges' metaphorically refers to feelings or emotions, not a physical or geographical location."
3971,hinges,hing,laiutama,NaN,in,"Täna tunneb Valeri Repson midagi sellist , mida ta juba ammu kogenud pole — tema hinges laiutab tiibu hoopis võimutunne .",no,"The term 'hinges' refers to an emotional or figurative state, not to a geographic or physical location."
5175,hinges,hinge,protesteerima,NaN,in,"Teadsin väga hästi , kuidas ta kõiki tähttähelisi ettekirjutusi põlates nendesamade plaanide vastu oma hinges protesteeris .",no,"The phrase 'hinges' was classified as not a location ('no') because it metaphorically refers to the person's inner feelings or soul, not a geographical location."
5290,hingest,hinge,saama,välja,el,Et saaks hingest välja .,no,"The word 'hingest' refers to the soul or spirit and not a location, so it was classified as 'no'."
9218,hinges,hinge,arenema,NaN,in,eks me vaikselt vast areneme oma hinges .,no,"The phrase 'hinges' refers to a metaphorical or emotional state, not a physical or geographic location, hence it is classified as 'no'."


In [20]:
df[idx1 & (df["form"].str.contains("koju"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
907,koju,kodu,pagema,NaN,adit,Seekord pistsid poisid jooksu ja pagesid lähemal elava sõbra juurde koju .,no,The term 'koju' refers to the concept of homecoming rather than a specific location.
6796,koju,kodu,reisima,NaN,adit,Asendusliige Janno Simm tuleb jahiga Euroopasse ja reisib sealt koju .,no,"The phrase 'koju' refers to the concept of 'home' as a personal or emotional place rather than a geographically specific location, so it was classified as not a location."


In [21]:
df[idx1 & (df["form"].str.contains("Liidus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
18,Liidus,liit,jooma,NaN,in,"Nõukogude Eestis elas üks nn raamaturahvas , kes seda maad külastanud türgi luuletaja Nazõm Hikmeti sõnul “ luges kõige rohkem luulet ja jõi kõige rohkem viina ” terves Nõukogude Liidus .",no,"The phrase 'Liidus' is not classified as a location because it is part of the name 'Nõukogude Liidus', which refers to an organization (Soviet Union) rather than a specific geographical location."
587,Liidus,Liidu,toimima,NaN,in,"Arvamustevahetust tervitades peame ometi tagasi lükkama ettepanekud , milles soovitatakse kasutada endises N Liidus toiminud suhtlemisskeeme ja -võtteid .",no,The word 'Liidus' refers to the former Soviet Union as an entity and not a specific physical location.
2044,Liidus,Liidu,ootama,NaN,in,Maalehe ja Põllumajandus-Tööstuskoja korraldatud konverentsi Aasta põllumees 2002 raames toimunud väitlus teemal Mis meid ootab Euroopa Liidus ?,no,"The phrase 'Liidus' refers to the European Union but is not used as a specific location in this context, hence it is classified as 'no'."
2734,Liidus,liit,opereerima,NaN,in,"Euroopa Liidus opereerib Falck suurimat kiirabiteenistust ja on ainuke kontsern , kes pakub erakorralise meditsiini teenust oma koduriigist väljaspool .",no,"The phrase 'Liidus' refers to 'the Union' (likely 'European Union'), which is not a specific geographical location but an entity or organization."
4210,Liidus,Liidud,seisma,NaN,in,"Seetõttu tuleb meil praegu endale väga selgelt ja ilma igasuguse roosamannata teadvustada , et nii Euroopa Liidus kui ka NATO-s seisab meil esimese ülesandena päevakorras enda maksmapanek , oma reviiri mahamärkimine .",no,"The phrase 'Liidus' refers to an organizational entity (European Union or NATO) and not a geographic location, so it is not classified as a location."
6597,Liidus,Liidu,liikuma,NaN,in,"Bulgaaria , Rumeenia ja Sloveenia on juba Euroopa Liidus ning NATOs , Horvaatia , Makedoonia , Bosnia- ja Herzegoviina liiguvad samas suunas .",no,"The phrase 'Liidus' could theoretically refer to an organization like the European Union, but in this context, it does not denote a specific geographical or physical location, so it was classified as not location ('no')."
8750,Liidust,Liidu,saama,välja,el,siis kui Eesti sai N Liidust välja oli mitu ärevat hetke .,no,"The phrase 'Liidust' refers to an organization or union, not a geographic or physical place."
8821,Liidust,Liit,saama,välja,el,"Ühe hinna järgi saavad odavat viina Euroopa Liidust välja sõitjad , teise järgi kallimat viina Euroopa Liitu sisse sõitjad .",no,The phrase 'Liidust' is not classified as a location because it refers to 'European Union' conceptually rather than a specific geographic location.


In [22]:
df[idx1 & (df["form"].str.contains("nimetus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
8620,nimetusse,nimetus,viskama,NaN,adit,Vanim poeg Sulev viskas kivi kaugele-kaugele mingisse nimetusse järve .,no,"The word 'nimetusse' describes the quality or characteristic of the lake being unnamed and not a specific geographical location, so it is classified as 'no'."


In [23]:
df[idx1 & (df["form"].str.contains("pimeda"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
8176,pimedast,pime,pääsema,välja,el,"Esialgu on veel vara ennustada , kas too rändur sealt pimedast enam välja pääsebki .",no,"The word 'pimedast' refers to darkness, not a specific location."


In [24]:
df[idx1 & (df["form"].str.contains("peaaegu-pimeduses"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
355,peaaegu-pimeduses,peaaegu-pimedus,kõndima,NaN,in,"Me kõndisime soojas peaaegu-pimeduses jaama poole ja ma aimasin laiduväärsushäbi ja võidurõõmuga ette , kuhu see kõndimine viib .",no,"The phrase 'peaaegu-pimeduses' translates to 'almost darkness' and refers to lighting conditions rather than a physical location, so it was classified as 'no'."


In [25]:
df[idx1 & (df["form"].str.contains("laag"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
241,merelaagrisse,merelaager,ootama,NaN,ill,Pirita vaba aja keskus ootab 10.-14. juunini 7-12aastasi lapsi uudishimulike kunstnike laagrisse ( osavõtutasu 250 krooni ) ning 29. juulist 2. augustini 7-14aastasi merelaagrisse .,no,The phrase 'merelaagrisse' refers to a camp or event and not a geographical location.


In [26]:
df[idx1 & (df["form"].str.contains("Kõvaketas"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
7201,Kõvaketast,kõvaketa,röövima,NaN,el,-> Kõvaketast röövib ca 9 giga .,no,"The phrase 'Kõvaketast' refers to 'hard drive' and is not a location, hence classified as 'no'."


In [27]:
df[idx1 & (df["form"].str.contains("rööbas"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
3517,rööbastesse,rööbas,pöörama,NaN,ill,"Pärast seda pöörab elu taas normaalsetesse rööbastesse , "" lausus Adlas .",no,"The phrase 'rööbastesse' refers to tracks or rails in a metaphorical sense, not a physical location."


In [28]:
df[idx1 & (df["form"].str.contains("käekot"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
3056,käekotist,käekott,varastama,NaN,el,"Kasutades kannatanu abitut seisundit , varastas kõrvaltoast käekotist 500",no,"The phrase 'käekotist' means 'from the handbag,' which refers to an object and not a location."


In [39]:
df[idx1 & (df["form"].str.contains("kirja"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
11,ajakirjandusest,ajakirjandus,kostma,NaN,el,"Nüüd , kui esimesed kired Eurovisiooni lauluvõistluse järel on vaibumas , kostab mitmete riikide ajakirjandusest hääli , et Melodi Grand Prix vöistlusreegleid tuleks muutma hakata .",no,"The term 'ajakirjandusest' refers to journalism or media and not a physical geographic location, hence it is classified as no."
503,nimekirja,nimekiri,pistma,NaN,adit,"Playstationile pääseb see , kes on end varakult nimekirja pistnud .",no,"The phrase 'nimekirja' refers to a list or record and is not a geographic or physical location, hence it is classified as not a location."
598,kirjanduses,kirjandus,kohama,NaN,in,"Hiljem olen erialases kirjanduses kohanud artikleid , tõsiseid biokeemilisi uurimusi , kus on täheldatud seost piimavalkude , karotiini ja ajurakkude töö vahel ning mu silme ette on kerkinud mitte just eriti isuäratav vaatepilt valgest vedelikust , kus ujuvad porganditükid ja herned ning mida katavad rohelised saared : tillivarred , sõstralehed , roheline sibul , porrulauk ja muu ollus , kuhu on nõrutatud üksjagu harrast usku ning naiivset , kuid ealeski – eladeski – mitte toksilist inimlootust .",no,"The word 'kirjanduses' refers to a subject matter (literature), not a physical location."
692,kirjanduses,kirjandus,jälgima,NaN,in,Ka kirjanduses jälgin ma väga tähelepanelikult moegurude soovitusi .,no,The phrase 'kirjanduses' refers to literature and not to a geographic location.
785,läikajakirjades,läikajakiri,ilutsema,NaN,in,Kosmeetikafirma Estee Lauder toodangut reklaamiva briti näitlejatari ja modelli Elizabeth Hurley pilt ilutseb alatihti läikajakirjades .,no,"The phrase 'läikajakirjades' refers to magazines, which are not physical or geographic locations, thus it is classified as 'no'."
926,lisanimekirja,lisanimekiri,lubama,NaN,adit,"Tallinna börsi noteerimiskomisjon lubas eile ASA Kindlustuse aktsiad börsi lisanimekirja , kuigi firma taotles oma aktsiate noteerimist põhinimekirjas .",no,"The phrase 'lisanimekirja' refers to a stock market term, not a geographical location."
1296,erastamisnimekirjast,erastamisnimekiri,saama,välja,el,"Mis seal ikka - meenutan sedagi , et koos Ando Keskkülaga saime tollase peaministri Mart Laari ja tema majandusministri Toivo Jürgensoni abiga soolalao erastamisnimekirjast välja , mis avas võimaluse teha soolalaost kultuuriasutus .",no,"The phrase 'erastamisnimekirjast' refers to a privatization list, which is a concept and not a location."
1512,ajakirjandusse,ajakirjandus,laskma,NaN,adit,"Lukas ütles , et lase asi ajakirjandusse , vaatame , mis juhtub .",no,"The word 'ajakirjandusse' refers to journalism or the press, which denotes a medium and not a physical or geographical location."
1544,ajakirjandusest,ajakirjandus,hankima,NaN,el,Poisid kasutavad rohkem nõustamist e-meili vahendusel ja hangivad informatsiooni ajakirjandusest .,no,"The phrase 'ajakirjandusest' refers to 'press/journalism' rather than a specific physical location, so it is classified as not a location ('no')."
1545,valimisnimekirjadesse,valimisnimekiri,pidama,NaN,ill,""" Rahvas teeb juba nalja , et kas sügisel hakkate kõik hooldekodusse häälte järgi jooksma , "" sõnas Leib , kes peab hooldekogu elanike valimisnimekirjadesse kandmist ebaõigeks .",no,"The phrase 'valimisnimekirjadesse' refers to 'election lists' which represent an abstract concept rather than a location, so it is classified as not a location ('no')."


In [7]:
df[idx1 & (df["form"].str.contains("pagendus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
7935,pagendusse,pagendus,pidama,NaN,adit,"Millegipärast enamik represseeritute ühinguid ei pea neid pagendusse läinuteks , vaid peab repressioonide eest põgenejateks .",no,The word 'pagendusse' refers to exile or banishment rather than a specific geographic location.


In [9]:
df[idx1 & (df["form"].str.contains("liin"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
1936,elektriliinist,elektriliin,varastama,NaN,el,"Elektrit varastasid "" vabrikandid "" naabertalu elektriliinist .",no,"The phrase 'elektriliinist' refers to an electrical line, which is a utility infrastructure rather than a location, hence it is classified as 'no'."
4485,liinilt,liin,väljuma,NaN,abl,Land Cruiseri legendi 50. aastapäeva tähistamiseks väljusid Toyotal liinilt nüüd kaks Land Cruiseri juubeliaasta erimudelit .,no,"The word 'liinilt' refers to a production line, which is not a geographical location."
7040,Eluliini,eluliin,pöörduma,NaN,adit,"2000. aastal pöördus Eluliini 4003 inimest , kellest 1000 olid valmis elust loobuma .",no,"The phrase 'Eluliini' does not indicate a geographical or physical location, but rather an organization or service; therefore, it is not considered a location."
8365,elektriliinidesse,elektriliin,looma,NaN,ill,"Eile varahommikul lõi äike elektriliinidesse Ida-Virumaal Püssi alajaama juures , mis jättis Eesti Energia teatel ligi tunniks ajaks vooluta AS Flexa puidutööstuse Viru-Nigulas ja paljud väiketarbijad .",no,"The phrase 'elektriliinidesse' refers to power lines, which are not a specific geographic or locational reference."
9355,eesliinile,eesliin,ronima,NaN,all,"“ Ta on tark , temaga on huvitav rääkida , ” kiidab inspektor ja lisab samas , et limonovlaste liider on kaval kuju , kes koordineerib tegevust ja juhatab oma jüngreid , kuid ise eesliinile ei roni .",no,The word 'eesliinile' refers to 'front line' in a metaphorical sense and is not denoting a geographic location.
9547,liinile,liin,pöörduma,tagasi,all,"Me lihtsalt pöördume tagasi sellelesamale lõputute vaidluste liinile ning tulemus on see , et me ei ehita valmis ühtegi objekti .",no,"The phrase 'liinile' refers to a 'line,' not a geographical location."


In [10]:
df[idx1 & (df["form"].str.contains("That"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
10,That's,That,lindistama,NaN,in,"Elvis , Scotty Moore ja Bill Black lindistasid "" That's all right "" -nimelise laulu Sun Recordsile 5. juulil 1954. aastal .",no,"The term 'That's' refers to part of a song title and not a location, so it is classified as no."


In [11]:
df[idx1 & (df["form"].str.contains("sfäär"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,explanation
32,erasfääri,erasfäär,liikuma,NaN,adit,"Vabanemisel liiguksid nad erasfääri , kus teatud valdkondades , alates teatud tasemest , valitseb tugev töökäte puudus .",no,"The phrase 'erasfääri' refers to a conceptual domain or private sphere rather than a physical location, so it is classified as not a location ('no')."
4369,mõjusfääri,mõjusfäär,liikuma,NaN,adit,Politseisiseste motivatsiooni- ja distsipliiniprobleemide tõttu kaotab riik kontrolli politsei üle ja see liigub organiseeritud kuritegelike struktuuride mõjusfääri .,no,"The phrase 'mõjusfääri' refers to a sphere of influence rather than an actual geographic or physical location, so it is classified as 'no'."
5256,sfääris,sfäär,ringlema,NaN,in,"Kuna olen naftat puurinud ja geoloogilistel ekspeditsioonidel käinud , kõik mu noorepõlve aastad on ju rännakute rõõmud , siis olen selles sfääris ringelnud .",no,"The phrase 'sfääris' was classified as not a location because it refers to a metaphorical sphere of activity or domain, not a physical or geographical location."
9278,ametisfääris,ametisfäär,vastutama,NaN,in,"Kas te jagate minu arusaamist , et ministrid ei vastuta ikkagi ( mitte nii , nagu Kalle Jürgenson lapsemeelselt väitis ) isiklike tegude eest , vaid vastutavad oma ametisfääris toimuva eest ?",no,"The phrase 'ametisfääris' was classified as 'no' because it refers to a conceptual sphere of responsibility, not a physical location."
9818,atmosfääris,atmosfäär,laiutama,NaN,in,Vene suurriiklus laiutab atmosfääris nagu eeter .,no,"The word 'atmosfääris' translates to 'in the atmosphere', which refers to a physical environment or layer rather than a specific location."


## Süntaks mõne näite puhul

In [9]:
from estnltk import Text
import estnltk
from estnltk_neural.taggers import StanzaSyntaxTagger

2025-12-11 10:53:08.957477: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-11 10:53:09.292749: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 10:53:09.364629: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-12-11 10:53:09.364648: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore 

In [11]:
stanza_tagger = StanzaSyntaxTagger(input_type='morph_extended', input_morph_layer='morph_extended')


In [16]:
text = Text("Vanim poeg Sulev viskas kivi kaugele-kaugele mingisse nimetusse järve .")
text.tag_layer('morph_extended')
stanza_tagger.tag( text )
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Vanim', [{'id': 1, 'lemma': 'vanim', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('super', 'super'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 2, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('poeg', [{'id': 2, 'lemma': 'poeg', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 4, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('Sulev', [{'id': 3, 'lemma': 'Sulev', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 2, 'deprel': 'appos', 'deps': '_', 'misc': '_'}]),
Span('viskas', [{'id': 4, 'lemma': 'viskama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('kivi', [{'id': 5, 'lemma': 'kivi', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 4, 'deprel': 'obj', 'deps': '_', 'misc': '_'}]),
Span('kaugele-kaugele', [{'id': 6, 'lemma': 'kaugele-kaugele', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 4, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('mingisse', [{'id': 7, 'lemma': 'mingi', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('ill', 'ill')]), 'head': 8, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('nimetusse', [{'id': 8, 'lemma': 'nimetus', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('adit', 'adit')]), 'head': 4, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('järve', [{'id': 9, 'lemma': 'järv', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 4, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 10, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 4, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [14]:
text = Text("Nõukogude Eestis elas üks nn raamaturahvas , kes seda maad külastanud türgi luuletaja Nazõm Hikmeti sõnul “ luges kõige rohkem luulet ja jõi kõige rohkem viina ” terves Nõukogude Liidus .")
text.tag_layer('morph_extended')
stanza_tagger.tag( text )
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Nõukogude', [{'id': 1, 'lemma': 'Nõukogu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('pl', 'pl'), ('gen', 'gen')]), 'head': 2, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('Eestis', [{'id': 2, 'lemma': 'Eesti', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('in', 'in')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('elas', [{'id': 3, 'lemma': 'elama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('üks', [{'id': 4, 'lemma': 'üks', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('nom', 'nom'), ('l', 'l')]), 'head': 6, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('nn', [{'id': 5, 'lemma': 'nn', 'upostag': 'Y', 'xpostag': 'Y', 'feats': OrderedDict([('nominal', 'nominal')]), 'head': 6, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('raamaturahvas', [{'id': 6, 'lemma': 'raamaturahvas', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 7, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 18, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('kes', [{'id': 8, 'lemma': 'kes', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('nom', 'nom')]), 'head': 18, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('seda', [{'id': 9, 'lemma': 'see', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('part', 'part')]), 'head': 10, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('maad', [{'id': 10, 'lemma': 'maa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 11, 'deprel': 'obj', 'deps': '_', 'misc': '_'}]),
Span('külastanud', [{'id': 11, 'lemma': 'külastama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('partic', 'partic'), ('past', 'past'), ('ps', 'ps')]), 'head': 13, 'deprel': 'acl', 'deps': '_', 'misc': '_'}]),
Span('türgi', [{'id': 12, 'lemma': 'türgi', 'upostag': 'G', 'xpostag': 'G', 'feats': OrderedDict(), 'head': 13, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('luuletaja', [{'id': 13, 'lemma': 'luuletaja', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 16, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('Nazõm', [{'id': 14, 'lemma': 'Nazõm', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 13, 'deprel': 'appos', 'deps': '_', 'misc': '_'}]),
Span('Hikmeti', [{'id': 15, 'lemma': 'Hikmeti', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 14, 'deprel': 'flat', 'deps': '_', 'misc': '_'}]),
Span('sõnul', [{'id': 16, 'lemma': 'sõna', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('ad', 'ad')]), 'head': 18, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('“', [{'id': 17, 'lemma': '“', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 18, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('luges', [{'id': 18, 'lemma': 'lugema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 6, 'deprel': 'acl:relcl', 'deps': '_', 'misc': '_'}]),
Span('kõige', [{'id': 19, 'lemma': 'kõige', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 20, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('rohkem', [{'id': 20, 'lemma': 

In [15]:
text = Text("-> Kõvaketast röövib ca 9 giga .")
text.tag_layer('morph_extended')
stanza_tagger.tag( text )
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('-', [{'id': 1, 'lemma': '-', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 4, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('>', [{'id': 2, 'lemma': '>', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 4, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('Kõvaketast', [{'id': 3, 'lemma': 'kõvaketa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('el', 'el')]), 'head': 4, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('röövib', [{'id': 4, 'lemma': 'röövima', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('pres', 'pres'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('ca', [{'id': 5, 'lemma': 'ca', 'upostag': 'Y', 'xpostag': 'Y', 'feats': OrderedDict([('nominal', 'nominal')]), 'head': 6, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('9', [{'id': 6, 'lemma': '9', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 7, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('giga', [{'id': 7, 'lemma': 'giga', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 4, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 8, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 4, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [17]:
text = Text("Paljudel moosipirukasõpradel on ühine mure : täidis voolab küpsemise käigus pirukast välja .")
text.tag_layer('morph_extended')
stanza_tagger.tag( text )
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Paljudel', [{'id': 1, 'lemma': 'palju', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('ad', 'ad')]), 'head': 2, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('moosipirukasõpradel', [{'id': 2, 'lemma': 'moosipirukasõber', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('ad', 'ad')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('on', [{'id': 3, 'lemma': 'olema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('pres', 'pres'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 2, 'deprel': 'cop', 'deps': '_', 'misc': '_'}]),
Span('ühine', [{'id': 4, 'lemma': 'ühine', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 5, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('mure', [{'id': 5, 'lemma': 'mure', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 2, 'deprel': 'nsubj:cop', 'deps': '_', 'misc': '_'}]),
Span(':', [{'id': 6, 'lemma': ':', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 8, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('täidis', [{'id': 7, 'lemma': 'täidis', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 8, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('voolab', [{'id': 8, 'lemma': 'voolama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('aux', 'aux'), ('indic', 'indic'), ('pres', 'pres'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 2, 'deprel': 'parataxis', 'deps': '_', 'misc': '_'}]),
Span('küpsemise', [{'id': 9, 'lemma': 'küpsemine', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 8, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('käigus', [{'id': 10, 'lemma': 'käigus', 'upostag': 'K', 'xpostag': 'K', 'feats': OrderedDict([('post', 'post')]), 'head': 9, 'deprel': 'case', 'deps': '_', 'misc': '_'}]),
Span('pirukast', [{'id': 11, 'lemma': 'pirukas', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('el', 'el')]), 'head': 8, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('välja', [{'id': 12, 'lemma': 'välja', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 8, 'deprel': 'compound:prt', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 13, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 2, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [18]:
text = Text("Kasutades kannatanu abitut seisundit , varastas kõrvaltoast käekotist 500")
text.tag_layer('morph_extended')
stanza_tagger.tag( text )
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Kasutades', [{'id': 1, 'lemma': 'kasutama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('ger', 'ger')]), 'head': 6, 'deprel': 'advcl', 'deps': '_', 'misc': '_'}]),
Span('kannatanu', [{'id': 2, 'lemma': 'kannatanu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 4, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('abitut', [{'id': 3, 'lemma': 'abitu', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('part', 'part')]), 'head': 4, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('seisundit', [{'id': 4, 'lemma': 'seisund', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 1, 'deprel': 'obj', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 5, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 6, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('varastas', [{'id': 6, 'lemma': 'varastama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('aux', 'aux'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('kõrvaltoast', [{'id': 7, 'lemma': 'kõrvaltuba', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('el', 'el')]), 'head': 6, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('käekotist', [{'id': 8, 'lemma': 'käekott', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('el', 'el')]), 'head': 6, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('500', [{'id': 9, 'lemma': '500', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 6, 'deprel': 'obj', 'deps': '_', 'misc': '_'}])])

## Kui palju saaks üldse ette anda gpt-le märgendamiseks

In [64]:
df4 = pd.read_csv("../gpt_input/n80_examples_large_v2.csv", encoding="utf-8", sep="|")

In [81]:
df4

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag
0,4379163,Moskvas,Moskva,süttima,NaN,in,"Moskvas süttis põlema juhtimisinstituut , hukkunud on 6 inimest ja haiglaravi vajab veel 35.",2729940,NaN,location,LOC
1,1374086,Suzhous,Suzhou,arenema,NaN,in,"Telemees veendus oma silmaga , et siiditööstus on Suzhous arenenud kuni tänaseni : "" Sealses Siidiinstituudis asuvas töökojas tehakse uskumatut tööd .",862982,NaN,location,LOC
2,1620478,kuus,kuu,kogunema,NaN,in,Viimasel veerandajal kogunes neid Magicule kuus ja lisaajal kaks .,1017872,NaN,NaN,NaN
3,11509956,regedele,regi,ronima,NaN,all,"Aga need kui need Esä poolt armastuse ja hoolega tehtud lumelauad ilusa kollase leegiga põlema lõid , meile kõigile veel pisku sooja pakkudes , enne kui me oma külmadele regedele ronime , viirastus mulle veel korra me mäeküljetare , kuidas see põles – ja ma mõtlesin , et kas saan veel kunagi tagasi sellesse kohta , kus see oli seisnud , ja ka üles mäekuplile endale ning kas siis on veel seal kidurate kadakate ringis alles Peko tammepuust kuju .",7155623,NaN,NaN,NaN
4,28045293,piiritusse,piiritus,panema,NaN,adit,loomaporno 1 : panen tallele loomaporno 2 : olin tõhus loomaporno 3 : olin sõprade seas loomaporno 4 : sain sealt võimlaporno : panin kitse hamstriporno : panin põske linnuporno : panin tihasele pekki astroporno : panin tähele naljamaia taimetoitlasest elektrikuporno : panin pirni tiigiporno suurte elukatega : panin konna piiritusse botaanikaporno : panin juurde köögiviljaporno : panin peeti hiinlase kriminaalporno : bhandi mindi wangi äiaporno : panin tütre mehele äiuporno : panin tütre magama koduarestis tüübi porno : panin ukse lukku läbipüksteporno : panin su luku taha tubliporno : hästi paneb !,18521383,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
75407,9778555,õllepoest,õllepood,kihutama,välja,el,"Ja isegi seda on mõned ringi rändavad sibid , kraavilõikajad ning teised tublid tööinimesed kõnelenud , et kui nad ka enesele puhtad linnariided selga ostavad ja säärased läikivad kamassid jalga löövad , mille pealt nemad oma larhvi nagu peeglist näevad , siis kirtsutavad linnasaksad ikka nina ja kihutavad vaesed mehed õllepoest välja .",6091834,NaN,location,NaN
75408,1821886,Ugalast,Ugala,minema,ära,el,""" Lahkusin siis , kui Jaan Tooming Ugalast ära läks , "" sõnab ta .",1146552,NaN,NaN,LOC
75409,310636,Davosis,Davos,toimuma,NaN,in,Esimest korda tuldi Davosis toimunud Maailma Majandusfoorumil välja nn valmisoleku indeksiga .,187090,NaN,location,LOC
75410,2545990,Muuseumis,Muuseumi,leiduma,NaN,in,Briti Muuseumis leidub kolm sääraste peekrite fragmenti .,1599871,NaN,NaN,ORG


In [70]:
aggreg2 = df4.groupby(["verb", "verb_compound", "morph_case"], dropna=False).agg(
        lemmas = ("lemma", lambda x :len(x))
).sort_values('lemmas', ascending=False)
aggreg2

,,,lemmas
verb,verb_compound,morph_case,
ajama,NaN,in,500
peetuma,NaN,in,500
panema,NaN,adit,500
paiknema,NaN,ad,500
ostma,NaN,el,500
...,...,...,...
saatma,edasi,ill,37
lendama,edasi,ill,35
turustama,NaN,in,35
